# Phase 13 — the annotators' spines as a second target

Phase 12 traced the 3,501 filaments the current configuration misses (pooled
five-fold PQ 0.4127). The largest group, 953, are long filaments covered by a
candidate that lies inside them and reaches about a third of their area; the
rest of such a filament has a median probability of 0.013. The model finds the
darkest stretch and stops where the annotators carry on. Recovering two thirds
of them would take the pooled PQ past 0.45.

Every annotation also carries a spine: a centre line drawn by hand from one end
of the filament to the other. The organisers allow it as auxiliary
supervision. This notebook trains HRNet-W32 at 1024 (run *e* of Phase 6b) with
a second output channel that learns the spines, and, on the other card, run *e*
itself once more as the control — the same configuration without spines, in
the same session. Inference reads only the first channel, so everything after
the network is unchanged.

Both are scored on fold 0 the way the submission is made: eight-view TTA, then
the Phase 10 post-processing (2048, threshold 0.40, minimum area 400, rejoining
16, score cut 0.65). Phase 12's breakdown of the misses says whether a change
in PQ comes from the long filaments it is aimed at.

**Adoption rule** (one fold): spines at least +0.01 over the control, with the
gain explained by fewer partly covered or fragmented misses. The control also
measures how far one retraining moves fold 0 from run *e*'s Phase 10 score,
0.4016 — the open question of Phase 7.

**Before running**, in the notebook settings:

1. Accelerator: **GPU T4 x2** (one run per card)
2. Internet: **on**
3. Inputs: the competition data
4. **Save Version** with *Save output* on

Expected wall clock: about two hours (training about 110 minutes, both runs at
once; maps and scoring about 15 minutes).

**Output.** Each run's checkpoint and logs under `spine/` and `control/`, its
eight-view fold 0 maps under `maps/`, and the scores and miss breakdowns under
`scores/`.

**Re-running after a failure.** Attach this notebook's earlier version as an
input. Finished training and maps are copied back and skipped.

## 1. Clone the repository and put it on the path

Cloned into `/tmp`, not `/kaggle/working`, so the clone does not count against
the 500 output files.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase13-spine"  # branch or commit hash
CHECKOUT = "/tmp/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; torch stays as it is.
!pip install -q "segmentation-models-pytorch>=0.5"

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import segmentation_models_pytorch as smp
import torch

import filament

print("filament", filament.__version__)
print("smp", smp.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count())

## 2. Point the package at the competition data

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

os.environ["MAGFILO_ROOT"] = str(candidates[0].parent.parent)
paths = load_paths().require_dataset()
print("MAGFILO_ROOT =", os.environ["MAGFILO_ROOT"])
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))

## 3. The two runs

In [ ]:
import json
import shutil
from dataclasses import replace

from filament.training.config import TrainConfig

WORKING = Path("/kaggle/working")
RUNS = {
    "spine": f"{CHECKOUT}/configs/phase13/e_unet_hrnet_spine.yaml",
    "control": f"{CHECKOUT}/configs/phase6/e_unet_hrnet.yaml",
}
configs = {
    name: replace(TrainConfig.from_yaml(path), num_workers=2, output_dir=WORKING / name)
    for name, path in RUNS.items()
}
for name, config in configs.items():
    print(
        f"{name:8s} {config.encoder} {config.image_size}px batch {config.batch_size} "
        f"epochs {config.epochs} spine_weight {config.loss.spine_weight} fold {config.fold}"
    )

# Kaggle empties /kaggle/working when a session starts, so an earlier version's
# work is only reachable through the inputs.
RUN_FILES = ("best.pt", "config.yaml", "history.json", "train.log")
for name, config in configs.items():
    config.output_dir.mkdir(parents=True, exist_ok=True)
    for source in Path("/kaggle/input").glob(f"**/{name}/best.pt"):
        for item in RUN_FILES:
            if (source.parent / item).exists() and not (config.output_dir / item).exists():
                shutil.copy(source.parent / item, config.output_dir / item)
        print(f"{name}: restored from {source.parent}")
        break
for folder, pattern in (("maps", "*.npz"), ("scores", "*")):
    (WORKING / folder).mkdir(exist_ok=True)
    for source in Path("/kaggle/input").glob(f"**/{folder}/{pattern}"):
        if source.is_file() and not (WORKING / folder / source.name).exists():
            shutil.copy(source, WORKING / folder / source.name)
            print(f"restored {folder}/{source.name}")

## 4. Training, one run per card

Each run is `scripts/train.py` in its own process, pinned to one card with
`CUDA_VISIBLE_DEVICES`, so that each seeds its own random state.

In [ ]:
import subprocess
import sys
import threading
import time

gpu_count = torch.cuda.device_count()
pending = [name for name in RUNS if not (configs[name].output_dir / "best.pt").exists()]
QUEUES = (
    {gpu: pending[gpu :: max(gpu_count, 1)] for gpu in range(max(gpu_count, 1))}
    if gpu_count
    else {}
)
print(f"{gpu_count} GPU(s):", QUEUES)
print_lock = threading.Lock()
failures = {}


def run_queue(gpu, names):
    # An exception in a thread is otherwise printed and forgotten, and the
    # notebook would go on to fail later on a missing checkpoint.
    try:
        _run_queue(gpu, names)
    except Exception as error:
        with print_lock:
            print(f"[gpu{gpu}] stopped: {error!r}")
        failures.update({name: repr(error) for name in names if name not in failures})
        raise


def _run_queue(gpu, names):
    for name in names:
        config = configs[name]
        command = [
            sys.executable,
            f"{CHECKOUT}/scripts/train.py",
            "--config",
            RUNS[name],
            "--output-dir",
            str(config.output_dir),
            "--num-workers",
            str(config.num_workers),
        ]
        environment = os.environ | {
            "CUDA_VISIBLE_DEVICES": str(gpu),
            "PYTHONPATH": f"{CHECKOUT}/src",
        }
        with print_lock:
            print(f"[gpu{gpu}] {name}: started")
        started = time.perf_counter()
        with open(config.output_dir / "train.log", "w") as log:
            process = subprocess.Popen(
                command,
                cwd=CHECKOUT,
                env=environment,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            for line in process.stdout:
                log.write(line)
                log.flush()
                if "Epoch" in line or "Error" in line or "Traceback" in line:
                    with print_lock:
                        print(f"[gpu{gpu}] {name}: {line.rstrip()}")
            exit_code = process.wait()
        with print_lock:
            minutes = (time.perf_counter() - started) / 60
            print(f"[gpu{gpu}] {name}: exit {exit_code} after {minutes:.1f} min")
        if exit_code != 0:
            failures[name] = exit_code


threads = [threading.Thread(target=run_queue, args=item, daemon=True) for item in QUEUES.items()]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
for name, exit_code in failures.items():
    tail = (configs[name].output_dir / "train.log").read_text().splitlines()[-20:]
    print(f"{name} FAILED with exit code {exit_code}:", *tail, sep="\n")
if failures:
    raise SystemExit(f"Training failed for {sorted(failures)}.")

## 5. Eight-view maps of fold 0

The same TTA as the submission: each frame in the eight flips and quarter
turns, turned back and averaged. Only the first output channel, the mask, is
read.

In [ ]:
from filament.data.image import load_grayscale
from filament.data.split import load_fold
from filament.evaluation import predict_probability_tta
from filament.postprocess.search import save_map_bundle
from filament.training.loop import HISTORY_NAME, load_checkpoint

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
val_stems = sorted(load_fold(0, paths.splits_dir).val)
for name, config in configs.items():
    path = WORKING / "maps" / f"{name}_fold0.npz"
    if path.exists():
        print(f"{name}: maps already written")
        continue
    model, stored = load_checkpoint(config.output_dir / "best.pt", device=DEVICE)
    started = time.perf_counter()
    maps = {
        stem: predict_probability_tta(
            model,
            load_grayscale(paths.train_images / f"{stem}.jpeg"),
            size=config.image_size,
            device=DEVICE,
        )
        for stem in val_stems
    }
    save_map_bundle(maps, path)
    history = json.loads((config.output_dir / HISTORY_NAME).read_text())
    best = min(history, key=lambda item: item["val_loss"])
    print(
        f"{name}: {len(maps)} maps in {time.perf_counter() - started:.0f}s; "
        f"best epoch {best['epoch']} of {len(history)}, val loss {best['val_loss']:.4f}"
    )
    del model, maps
    torch.cuda.empty_cache()

## 6. Scores and the misses

`scripts/sweep_maps.py` scores each run's maps under the Phase 10 setting and,
for reference, the same without the score cut; `scripts/miss_breakdown.py`
sorts every annotated filament of fold 0 by outcome, as in Phase 12. Both run
as their own processes on the CPU.

In [ ]:
import pandas as pd

from filament.metrics.misses import CATEGORIES
from filament.postprocess.search import SweepResult

PHASE10 = {
    "resolution": 2048,
    "threshold": 0.4,
    "min_area": 400,
    "join_gap": 16.0,
    "join_offset": 12.0,
    "min_score": 0.65,
}
SETTINGS = [PHASE10, PHASE10 | {"min_score": 0.0}]
# Run e on fold 0 under the same TTA and post-processing (Phase 10, from Phase 9's maps).
RUN_E_FOLD0 = {"pq": 0.4016}
scores_dir = WORKING / "scores"
(scores_dir / "settings.json").write_text(json.dumps(SETTINGS))
(scores_dir / "phase10.json").write_text(json.dumps(PHASE10))
environment = os.environ | {"PYTHONPATH": f"{CHECKOUT}/src"}

jobs = []
for name in configs:
    maps = str(WORKING / "maps" / f"{name}_fold0.npz")
    sweep_out, miss_out = scores_dir / f"{name}_sweep.json", scores_dir / f"{name}_misses.csv"
    if not sweep_out.exists():
        jobs.append(
            subprocess.Popen(
                [
                    sys.executable,
                    f"{CHECKOUT}/scripts/sweep_maps.py",
                    "--maps",
                    maps,
                    "--fold",
                    "0",
                    "--settings",
                    str(scores_dir / "settings.json"),
                    "--output",
                    str(sweep_out),
                ],
                cwd=CHECKOUT,
                env=environment,
            )
        )
    if not miss_out.exists():
        jobs.append(
            subprocess.Popen(
                [
                    sys.executable,
                    f"{CHECKOUT}/scripts/miss_breakdown.py",
                    "--maps",
                    maps,
                    "--fold",
                    "0",
                    "--setting",
                    str(scores_dir / "phase10.json"),
                    "--output",
                    str(miss_out),
                ],
                cwd=CHECKOUT,
                env=environment,
            )
        )
if any(job.wait() for job in jobs):
    raise SystemExit("A scoring process failed; its output is above.")

rows = {}
for name in configs:
    sweep = SweepResult.from_records(
        json.loads((scores_dir / f"{name}_sweep.json").read_text())["points"]
    )
    misses = pd.read_csv(scores_dir / f"{name}_misses.csv")
    for point in sweep.points:
        label = f"{name}, score cut {point.setting.values['min_score']}"
        rows[label] = {
            "pq": point.pq.pq,
            "sq": point.pq.sq,
            "rq": point.pq.rq,
            "tp": point.pq.tp,
            "fp": point.pq.fp,
            "fn": point.pq.fn,
        }
    counts = misses["outcome"].value_counts()
    rows[f"{name}, score cut 0.65"] |= {
        category: int(counts.get(category, 0)) for category in CATEGORIES
    }
table = pd.DataFrame(rows).T
spine, control = rows["spine, score cut 0.65"], rows["control, score cut 0.65"]
gain = spine["pq"] - control["pq"]
sq_gain, rq_gain = spine["sq"] - control["sq"], spine["rq"] - control["rq"]
print(f"spine - control: PQ {gain:+.4f} (SQ {sq_gain:+.4f}, RQ {rq_gain:+.4f})")
rerun = control["pq"] - RUN_E_FOLD0["pq"]
print(f"control - run e (Phase 10, fold 0): PQ {rerun:+.4f}  <- what one retraining moves")
for category in ("too_small", "fragmented", "unseen", "below_threshold", "too_large"):
    print(
        f"  {category:16s} spine {spine[category]:4d}  control {control[category]:4d}  "
        f"change {spine[category] - control[category]:+d}"
    )
print("ADOPT for five folds" if gain >= 0.01 else "below the +0.01 bar for one fold")
(scores_dir / "summary.json").write_text(
    json.dumps({"rows": rows, "gain": gain, "commit": REF}, indent=1)
)
table.round(4)

In [ ]:
# Kaggle keeps at most 500 output files and drops the rest without an error.
saved = [item for item in WORKING.rglob("*") if item.is_file()]
total_mb = sum(item.stat().st_size for item in saved) / 1e6
print(f"{len(saved)} files in /kaggle/working, {total_mb:.0f} MB")
if len(saved) > 500:
    raise SystemExit(f"{len(saved)} output files; Kaggle would silently drop some of them.")

## 7. What to record

- both runs' PQ, SQ, RQ, TP, FP, FN under the Phase 10 setting, with and
  without the score cut, and their best epoch and validation loss
- the miss breakdown of both, and the change in partly covered and fragmented
  misses
- the control against run *e*'s 0.4016: what one retraining moves fold 0
- the commit this notebook cloned